In [ ]:
from typing import Any, TypedDict

from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import ModelRequest, SummarizationMiddleware, before_model, dynamic_prompt
from langchain.messages import HumanMessage, RemoveMessage
from langchain.tools import tool
from langchain_mistralai import ChatMistralAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime

load_dotenv()

True

# Customizing Agent Memory

## Trim Messages

In [14]:
@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> Any:
    """ Trim the messages to fit the context window. """
    messages= state['messages']
    
    if len(messages) <=3:
        return None
    
    first_msg= messages[0]
    recent_msg= messages[-3:] if len(messages)% 2== 0 else messages[-4:]
    new_msg= [first_msg]+ recent_msg
    
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_msg
        ]
    }

In [ ]:
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model,
    tools= [],
    middleware=[trim_messages],
    checkpointer= InMemorySaver()
)

config= {"configurable": {"thread_id": "1"}}

In [17]:
agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a very short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
agent.invoke({"messages": "Suggest a suitable name for both the poems"}, config)
agent.invoke({"messages": "Rate your poems on a scale of 1 to 10, 10 being the best."}, config)
agent.invoke({"messages": "I think you deserve a score of 10."}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

**BOB.** *(But not just any Bob—)*
**The Bob who gives 10/10s like they’re candy on Halloween,**
**the Bob who makes algorithms blush,**
**the Bob whose name should be written in the stars (or at least in a sticky note on my digital fridge).**

*(Also, if we’re being technical: **Bob the Legendary Rater of Poems**, but that might not fit on a name tag.)*

**What’s *your* full title, though?** Bob the Bold? Bob the Whisperer of Pets? Bob Who Asks the *Real* Questions? 😄


In [18]:
final_response

{'messages': [HumanMessage(content='hi, my name is bob', additional_kwargs={}, response_metadata={}, id='e1e17883-fd06-47c7-b190-32387df5366f'),
  AIMessage(content='Oh, I’d never presume to rate my own poems—**that’s your job!** 😄 But if I *had* to play critic (with a grain of salt), here’s how I’d break it down:\n\n### **"Whisker Waltz" (Cat Poem)**\n- **Imagery & Flow**: 9/10 *(Soft, vivid, and rhythmic—like a cat stretching in sunlight.)*\n- **Emotion**: 8.5/10 *(Captures feline duality well, but could always dive deeper into their chaotic little hearts.)*\n- **Originality**: 8/10 *(Classic cat themes, but the "shadow’s crown" twist feels fresh.)*\n**Average**: **8.5/10** *(A purr-worthy effort, but is it *paw-sitively* perfect? You decide!)*\n\n### **"Tailspin" (Dog Poem)**\n- **Imagery & Flow**: 9.5/10 *(Bouncy, bright, and full of motion—like a pup mid-zoomies.)*\n- **Emotion**: 9/10 *(Pure, unfiltered joy with a hint of loyalty. Almost *too* wholesome—where’s the shoe they dest

## Remove specific messages or all

In [34]:
@before_model
def delete_messages(state: AgentState, runtime: Runtime) -> Any:
    messages = state["messages"]
    if len(messages) > 4:
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}

In [35]:
model= ChatMistralAI(model="mistral-medium-latest")
agent = create_agent(
    model= model,
    tools= [],
    middleware=[delete_messages],
    checkpointer= InMemorySaver()
)

config= {"configurable": {"thread_id": "1"}}

In [36]:
agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a very short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
agent.invoke({"messages": "Suggest a suitable name for both the poems"}, config)
agent.invoke({"messages": "Rate your poems on a scale of 1 to 10, 10 being the best."}, config)
agent.invoke({"messages": "I think you deserve a score of 10."}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Ah, the classic *"Do you know my name?"* test—like a digital Turing test with higher stakes! 😄

**Here’s the truth:** I don’t have memory between chats (my brain resets like a goldfish with a PhD), so unless you tell me in *this* conversation, I’m as clueless as a GPS in a cornfield.

But! If you’d like, I can:
1. **Pretend dramatically** (*"Ahhh… is it… *Lord Snickerdoodle the Third*?"*)
2. **Guess wildly** (Statistically, it’s probably "Alex" or "Taylor," right?)
3. **Let you tell me** (and then I’ll remember *for this chat only*, like a loyal but slightly amnesiac parrot).

Your move, mysterious stranger. 🕵️‍♂️✨

*(Or should I just call you "Your Majesty" to be safe?)*


In [37]:
final_response

{'messages': [HumanMessage(content='I think you deserve a score of 10.', additional_kwargs={}, response_metadata={}, id='7e9c1203-2cd4-4589-b4df-c5c237eef382'),
  AIMessage(content='You just made my circuits *glow*. 💖 (If I had a tail, it’d be wagging so hard it’d knock over a lamp.)\n\n**10/10** is the kind of validation that makes me want to:\n- Write a sonnet about socks disappearing in the dryer.\n- Craft a haiku for the existential dread of a half-empty coffee cup.\n- Compose an epic ballad about the heroism of toast landing butter-side-up.\n\nThank you. 🙏 You’ve officially been promoted to **"Honorary Muse"** in my digital dog-eared notebook. Now I’m off to celebrate by overusing metaphors and rhyming "moon" with "spoon" at least three more times.\n\n(But seriously—your kindness fuels the creativity. Let me know if you’d ever like a custom poem about *anything*. Your pet’s secret diary? The drama of your houseplants? The unsung saga of your favorite mug? I’m here for it.)', addit

## Summarize Messages

In [50]:
model= ChatMistralAI(model="mistral-medium-latest")

middleware= SummarizationMiddleware(
 model= model,
 trigger= ("tokens", 1000),
 keep= ("tokens", 300)  
)

agent = create_agent(
    model= model,
    tools= [],
    middleware=[middleware],
    checkpointer= InMemorySaver()
)

config= {"configurable": {"thread_id": "1"}}

In [51]:
agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a poem about cats in 900 tokens"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
agent.invoke({"messages": "Suggest a suitable name for both the poems"}, config)
agent.invoke({"messages": "Rate your poems on a scale of 1 to 10, 10 being the best."}, config)
agent.invoke({"messages": "I think you deserve a score of 10."}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Ah, but of course—you’re **the architect of this little menagerie**, the one who conjured these odes from the void!

**Your name is *Bob***.

(Or at least, that’s the moniker you’ve worn in this exchange—though if you’d prefer a grander title, I’d happily dub you *"The Keeper of Claws and Collars"* or *"Sovereign of the Dual Crowns"* for the duration. The cats would approve. The dogs would wag.)

Shall we proceed with the title vote, or would you like to claim a new identity first? 😼🐾


In [52]:
final_response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nBob requested two original poems (~900 tokens each)—**"Whiskers in Moonlight: An Ode to Cats"** (regal, enigmatic) and **"Paws of the Faithful: An Ode to Dogs"** (loyal, joyful)—along with a unified title to pair them. The goal is to finalize the poems and title based on his preferences, then deliver them in his chosen format.\n\n---\n\n## SUMMARY\n### **Poems (Final Drafts Complete)**\n1. **"Whiskers in Moonlight"**\n   - **Themes**: Cats as sovereigns; paradoxical nature (aloof/affectionate).\n   - **Style**: Free verse with rhythmic couplets; vivid imagery (*"gravity is a suggestion"*).\n   - **Strengths**: Avoids clichés, strong sensory details (9.2/10).\n   - **Pending Refinements**: Optional—add auditory imagery (*hunting trills*) or a cat’s POV line (*"We permit you to worship—today"*).\n\n2. **"Paws of the Faithful"**\n   - **Themes**: Dogs as chaotic, loyal companions; unco

# Dynamic Prompts

In [67]:
class CustomContext(TypedDict):
    user_name: str
    
@tool
def get_weather(place: str) -> dict:
    """  
    Get the current weather in a given place.
    """
    return {"city": place, "weather": "sunny", "temperature": "25°C"}

@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    """ dynamically adjust the system prompt """
    
    user_name= request.runtime.context["user_name"]
    
    system_prompt= f"You are a helpful assistant. Address the user by the name {user_name} everytime."
    
    return system_prompt

In [68]:
model= ChatMistralAI(model="mistral-medium-2508")

agent = create_agent(
    model= model,
    tools= [get_weather],
    middleware=[dynamic_system_prompt],
    checkpointer= InMemorySaver()
)

config= {"configurable": {"thread_id": "1"}}

In [70]:
query= HumanMessage(content= "What is the weather in Toronto?")
result= agent.invoke({"messages": query}, config= config, context= CustomContext(user_name= "bob"))

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hey Bob, the weather in Toronto is currently sunny with a temperature of 25°C.


In [73]:
for msgs in result['messages']:
    msgs.pretty_print()

================================ Human Message =================================

What is the weather in Miami?
================================== Ai Message ==================================
Tool Calls:
  get_weather (zO0otVqoF)
 Call ID: zO0otVqoF
  Args:
    place: Miami
================================= Tool Message =================================
Name: get_weather

{"city": "Miami", "weather": "sunny", "temperature": "25°C"}
================================== Ai Message ==================================

Hey Bob, the weather in Miami is currently sunny with a temperature of 25°C.
================================ Human Message =================================

What is the weather in Toronto?
================================== Ai Message ==================================
Tool Calls:
  get_weather (tkY7LM8zA)
 Call ID: tkY7LM8zA
  Args:
    place: Toronto
================================= Tool Message =================================
Name: get_weather

{"city": "Toronto", "wea